In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
matriz = pd.read_csv('../datasets/matriz_produto_usuario.csv')

In [3]:
matriz

,id_produto,1,2,3,4,5,6,7,8,9,...,491,492,493,494,495,496,497,498,499,500
0,1,0.000000,0.000000,0.000000,0.000000,0.0000,1.343249,1.564607,1.103960,0.000000,...,0.000000,0.000000,1.649123,0.000000,1.670300,1.000000,0.000000,0.000000,1.572093,1.782895
1,2,0.000000,1.024570,0.000000,2.000000,0.0000,0.000000,1.671348,1.678218,0.000000,...,5.016173,0.000000,1.087719,1.477273,0.000000,0.000000,5.234637,0.000000,0.000000,0.000000
2,3,0.000000,0.000000,0.000000,0.000000,2.0325,1.228833,1.000000,3.596535,0.000000,...,0.000000,0.000000,1.288221,0.000000,0.000000,0.000000,0.000000,3.000000,3.234884,0.000000
3,4,0.000000,1.393120,1.789116,0.000000,0.0000,6.121281,0.000000,1.970297,0.000000,...,1.000000,2.967816,0.000000,0.000000,0.000000,1.707165,0.000000,0.000000,0.000000,1.000000
4,5,0.000000,1.206388,0.000000,1.663333,0.0000,0.000000,1.404494,1.034653,3.129518,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.310056,0.000000,0.000000,0.000000
5,6,0.000000,1.272727,0.000000,0.000000,0.0000,1.466819,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,1.343324,3.333333,2.000000,1.772959,4.397674,0.000000
6,7,1.739857,2.909091,0.000000,0.000000,0.0000,2.000000,1.772472,0.000000,1.746988,...,0.000000,0.000000,2.000000,0.000000,2.463215,0.000000,1.298883,0.000000,1.646512,0.000000
7,8,0.000000,0.000000,0.000000,0.000000,1.8725,0.000000,0.000000,0.000000,0.000000,...,1.070081,1.342529,0.000000,0.000000,0.000000,2.000000,0.000000,0.000000,1.300000,0.000000
8,9,0.000000,1.167076,1.000000,0.000000,1.6750,3.327231,2.000000,0.000000,1.852410,...,0.000000,1.144828,0.000000,0.000000,1.697548,0.000000,0.000000,0.000000,1.976744,0.000000
9,10,0.000000,0.000000,0.000000,0.000000,5.2775,0.000000,2.907303,0.000000,1.831325,...,0.000000,0.000000,0.000000,2.700758,1.324251,0.000000,1.687151,0.000000,0.000000,0.000000


In [4]:
ids_produtos = matriz['id_produto'].values
X_df = matriz.drop(columns='id_produto')

In [5]:
X = X_df.to_numpy(dtype=float)
X_sparse = csr_matrix(X)

In [6]:
n_produtos, n_usuarios = X.shape
total_celulas = n_produtos * n_usuarios
qtd_zeros = (X == 0).sum()
esparsidade_global = qtd_zeros / total_celulas if total_celulas > 0 else np.nan

In [7]:
compradores_por_produto = (X != 0).sum(axis=1)

In [8]:
tabela_compradores_produto = pd.DataFrame({
    'id_produto': ids_produtos,
    'n_usuarios_compradores': compradores_por_produto
}).sort_values('id_produto').reset_index(drop=True)

In [9]:
estatisticas_compradores_produto = pd.Series(compradores_por_produto).describe(percentiles=[0.25, 0.5, 0.75])
resumo_compradores_produto = pd.DataFrame({
    'minimo': [estatisticas_compradores_produto['min']],
    'q1': [estatisticas_compradores_produto['25%']],
    'mediana': [estatisticas_compradores_produto['50%']],
    'media': [estatisticas_compradores_produto['mean']],
    'q3': [estatisticas_compradores_produto['75%']],
    'maximo': [estatisticas_compradores_produto['max']]
})

In [10]:
matriz_binaria = (X != 0).astype(int)
cooc_item_item = matriz_binaria @ matriz_binaria.T

mask_fora_diagonal = ~np.eye(n_produtos, dtype=bool)
cooc_fora_diagonal = cooc_item_item[mask_fora_diagonal]

In [11]:
if cooc_fora_diagonal.size > 0:
    serie_cooc = pd.Series(cooc_fora_diagonal)
    estatisticas_cooc = serie_cooc.describe(percentiles=[0.25, 0.5, 0.75])
    resumo_cooc_item_item = pd.DataFrame({
        'minimo': [estatisticas_cooc['min']],
        'q1': [estatisticas_cooc['25%']],
        'mediana': [estatisticas_cooc['50%']],
        'media': [estatisticas_cooc['mean']],
        'q3': [estatisticas_cooc['75%']],
        'maximo': [estatisticas_cooc['max']]
    })
else:
    resumo_cooc_item_item = pd.DataFrame({
        'minimo': [np.nan],
        'q1': [np.nan],
        'mediana': [np.nan],
        'media': [np.nan],
        'q3': [np.nan],
        'maximo': [np.nan]
    })

In [12]:
indices_upper = np.triu_indices(n_produtos, k=1)
cooc_upper = cooc_item_item[indices_upper]

In [13]:
pares_cooc_zero = int((cooc_upper == 0).sum())
pares_cooc_menor_5 = int((cooc_upper < 5).sum())
pares_cooc_menor_10 = int((cooc_upper < 10).sum())
pares_cooc_menor_20 = int((cooc_upper < 20).sum())
total_pares_produtos = len(cooc_upper)

In [14]:
n_neighbors_ajustado = min(6, n_produtos)

In [15]:
modelo_knn = NearestNeighbors(
    n_neighbors=n_neighbors_ajustado,
    metric='cosine',
    algorithm='brute'
)

In [16]:
modelo_knn.fit(X_sparse)

distancias, indices = modelo_knn.kneighbors(X_sparse)

similaridade_item_item = cosine_similarity(X_sparse)

similaridade_sem_diagonal = similaridade_item_item.copy()
np.fill_diagonal(similaridade_sem_diagonal, np.nan)
sim_fora_diagonal = similaridade_sem_diagonal[~np.isnan(similaridade_sem_diagonal)]

In [17]:
if sim_fora_diagonal.size > 0:
    serie_sim = pd.Series(sim_fora_diagonal)
    estatisticas_sim = serie_sim.describe(percentiles=[0.25, 0.5, 0.75])
    resumo_similaridade_item_item = pd.DataFrame({
        'minimo': [estatisticas_sim['min']],
        'q1': [estatisticas_sim['25%']],
        'mediana': [estatisticas_sim['50%']],
        'media': [estatisticas_sim['mean']],
        'q3': [estatisticas_sim['75%']],
        'maximo': [estatisticas_sim['max']]
    })
else:
    resumo_similaridade_item_item = pd.DataFrame({
        'minimo': [np.nan],
        'q1': [np.nan],
        'mediana': [np.nan],
        'media': [np.nan],
        'q3': [np.nan],
        'maximo': [np.nan]
    })

In [18]:
linhas_vizinhos = []

for i in range(n_produtos):
    id_produto_ref = ids_produtos[i]
    vizinhos_idx = indices[i]
    vizinhos_dist = distancias[i]
    vizinhos_sim = 1 - vizinhos_dist

    pares = []
    for idx_vizinho, sim in zip(vizinhos_idx, vizinhos_sim):
        if idx_vizinho != i:
            pares.append((ids_produtos[idx_vizinho], sim))

    pares = sorted(pares, key=lambda x: x[1], reverse=True)[:5]

    for rank, (id_vizinho, sim) in enumerate(pares, start=1):
        linhas_vizinhos.append({
            'id_produto': id_produto_ref,
            'rank_vizinho': rank,
            'id_produto_vizinho': id_vizinho,
            'similaridade': sim
        })

tabela_5_vizinhos = pd.DataFrame(linhas_vizinhos)

In [19]:
if tabela_5_vizinhos.empty:
    produtos_sem_vizinhos_uteis = pd.DataFrame({
        'id_produto': ids_produtos,
        'motivo': 'sem vizinhos calculados'
    })
else:
    limiar_sim_util = 0.01

    max_sim_por_produto = (
        tabela_5_vizinhos.groupby('id_produto', as_index=False)['similaridade']
        .max()
        .rename(columns={'similaridade': 'max_similaridade_vizinho'})
    )

    produtos_sem_vizinhos_uteis = pd.DataFrame({'id_produto': ids_produtos}).merge(
        max_sim_por_produto,
        on='id_produto',
        how='left'
    )

    produtos_sem_vizinhos_uteis['max_similaridade_vizinho'] = produtos_sem_vizinhos_uteis['max_similaridade_vizinho'].fillna(0)
    produtos_sem_vizinhos_uteis = produtos_sem_vizinhos_uteis[
        produtos_sem_vizinhos_uteis['max_similaridade_vizinho'] <= limiar_sim_util
    ].copy()

    produtos_sem_vizinhos_uteis['motivo'] = 'todas as similaridades sao zero ou muito proximas de zero'

In [20]:
proporcao_produtos_ate_2_compradores = (compradores_por_produto <= 2).mean() if n_produtos > 0 else np.nan
proporcao_pares_cooc_zero = (cooc_upper == 0).mean() if total_pares_produtos > 0 else np.nan
proporcao_pares_cooc_menor_5 = (cooc_upper < 5).mean() if total_pares_produtos > 0 else np.nan
proporcao_sim_baixa = (sim_fora_diagonal <= 0.01).mean() if sim_fora_diagonal.size > 0 else np.nan
proporcao_produtos_com_vizinho_util = 1 - (len(produtos_sem_vizinhos_uteis) / n_produtos) if n_produtos > 0 else np.nan

In [21]:
diagnosticos = []

if not np.isnan(proporcao_produtos_ate_2_compradores) and proporcao_produtos_ate_2_compradores >= 0.5:
    diagnosticos.append('Baixa robustez: a maioria dos produtos tem poucos usuarios compradores.')

if not np.isnan(proporcao_pares_cooc_zero) and proporcao_pares_cooc_zero >= 0.5:
    diagnosticos.append('Coocorrencia fraca: muitos pares de produtos nunca aparecem juntos.')

if not np.isnan(proporcao_pares_cooc_menor_5) and proporcao_pares_cooc_menor_5 >= 0.8:
    diagnosticos.append('Coocorrencia limitada: a maior parte dos pares tem menos de 5 usuarios em comum.')

if not np.isnan(proporcao_sim_baixa) and proporcao_sim_baixa >= 0.8:
    diagnosticos.append('Baixo poder discriminativo: as similaridades entre itens sao quase sempre proximas de zero.')

if (
    not np.isnan(proporcao_produtos_ate_2_compradores) and
    not np.isnan(proporcao_pares_cooc_zero) and
    not np.isnan(proporcao_sim_baixa) and
    not np.isnan(proporcao_produtos_com_vizinho_util) and
    proporcao_produtos_ate_2_compradores < 0.5 and
    proporcao_pares_cooc_zero < 0.5 and
    proporcao_sim_baixa < 0.8 and
    proporcao_produtos_com_vizinho_util >= 0.7
):
    diagnosticos.append('A abordagem parece viavel: ha cobertura razoavel por produto e vizinhos com similaridade positiva.')

if len(diagnosticos) == 0:
    diagnosticos.append('Diagnostico intermediario: a matriz pode ser utilizavel, mas a qualidade da vizinhanca item-item depende de ajustes de filtro e volume de interacoes.')

diagnostico_final = ' '.join(diagnosticos)

In [22]:
print('Esparsidade global:', esparsidade_global)
print('\nResumo compradores por produto:')
print(resumo_compradores_produto)
print('\nResumo coocorrencia item-item:')
print(resumo_cooc_item_item)
print('\nResumo similaridade item-item:')
print(resumo_similaridade_item_item)
print('\nPares com coocorrencia zero:', pares_cooc_zero)
print('Pares com coocorrencia < 5:', pares_cooc_menor_5)
print('Pares com coocorrencia < 10:', pares_cooc_menor_10)
print('Pares com coocorrencia < 20:', pares_cooc_menor_20)
print('\nDiagnostico final:')
print(diagnostico_final)


Esparsidade global: 0.6059

Resumo compradores por produto:
   minimo     q1  mediana   media      q3  maximo
0   178.0  193.5    197.0  197.05  203.25   215.0

Resumo coocorrencia item-item:
   minimo    q1  mediana      media    q3  maximo
0    59.0  72.0     78.0  77.994737  83.0   101.0

Resumo similaridade item-item:
     minimo        q1   mediana     media       q3    maximo
0  0.245937  0.301856  0.325873  0.324591  0.34657  0.404091

Pares com coocorrencia zero: 0
Pares com coocorrencia < 5: 0
Pares com coocorrencia < 10: 0
Pares com coocorrencia < 20: 0

Diagnostico final:
A abordagem parece viavel: ha cobertura razoavel por produto e vizinhos com similaridade positiva.
